# Pull new metrics from mccray
First based off 'Original Transcript'

In [1]:
import pandas as pd
import re, nltk, csv
from nltk.corpus import words, PlaintextCorpusReader

In [ ]:
# pull entire dataset
# df = pd.read_csv("../data/mccray/october_sprint/mccray d_main.csv")

df = pd.read_csv("../data/mccray/october_sprint/D_mcc_main_v2.csv")
print(f"df len: {len(df)}")
print(list(df))

# # nltk base
# # # nltk.download('words')
# english_vocab = set(w.lower() for w in words.words())

# improved ntlk corpus
wordslist = PlaintextCorpusReader('../data/mccray', 'combined_words_corpus.txt')
english_vocab = set(w.lower() for w in wordslist.words())

def save_csv(df, title):
    df.to_csv(title,
          index=False,
          encoding='utf-8',
          quoting=csv.QUOTE_NONNUMERIC,   
          escapechar='\\',
          lineterminator='\n')

print(len(df))

df len: 14427
['Title', 'Creator', 'Contributors', 'Date', 'Approximate Date', 'Source', 'Subject', 'Local Subject', 'S.C. County', 'Description', 'Extent', 'Digital Collection', 'Website', 'Contributing Institution', 'Rights', 'Time Period', 'Geographic Location', 'Language', 'Digitization Specifications', 'Date Digital', 'Type', 'Format', 'Media Type', 'Identifier', 'Note', 'Digital Assistant', 'OCLC number', 'Date created', 'Date modified', 'Reference URL', 'CONTENTdm number', 'CONTENTdm file name', 'CONTENTdm file path', 'Year', 'Original Transcript', 'Original Len', 'Special Pattern', 'General Pattern', 'Repeat Chars', 'Short/No Transcript', 'Quality', 'Issue Types', 'Detected Artifacts', 'Semi-clean Transcript', '% English (base ntlk)', '% Semi-clean English (base ntlk)', '% English', '% Semi-clean English']
14427


In [3]:
og_tr = "Original Transcript"
sc_tr = "Semi-clean Transcript"

title = 'Title'
descrip = 'Description'

# Function to apply re pattern
def re_pattern(text, pattern):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    total = len(text)
    matches = len(pattern.findall(text))
    return (matches / total) * 100

# patterns
# alphanum_standard = re.compile(r'[a-zA-Z0-9\s.,!?;:\'"()\-_/]')
alphanum_standard = re.compile(r'[a-zA-Z0-9\s!-/:-@[-`{-~]')
alpha_only = re.compile(r'[a-zA-Z]')

# nltk en
def percent_real_english_nltk(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    tokens = re.findall(r"[A-Za-z]{2,}", text.lower())
    if not tokens:
        return 0
    real = [t for t in tokens if t in english_vocab]
    return 100 * len(real) / len(tokens)

def english_nltk_metrics(text, vocab):
    """Return dict showing which tokens are recognized or not (NLTK-style regex)."""
    if not isinstance(text, str) or not text.strip() or not isinstance(text, str) or len(text) == 0:
        return {"words": [], "not_words": [], "word_count": 0, "valid_word_count": 0, "percentage": 0}
    tokens = re.findall(r"[A-Za-z]{2,}", text.lower())
    word_count = len(tokens)
    valid_words, not_words = [], []
    total_chars_count_from_valid_words = 0
    for t in tokens:
        if t in vocab:
            valid_words.append(t)
            total_chars_count_from_valid_words = total_chars_count_from_valid_words + len(t)
        else:
            found_variant = False
            if len(t) > 3:
                # Handle plural forms like "stories" -> "story" 
                if t.endswith("ies") or t.endswith("iest"): 
                    if t.endswith("ies"):
                        candidate = t[:-3] + "y"
                    else:
                        candidate = t[:-4] + "y"
                    if candidate in vocab :
                        valid_words.append(t)
                        found_variant = True
                # Only check other trims if no variant found yet
                if not found_variant:
                    for i in range(1, 4):  # remove 1–3 chars
                        base = t[:-i]
                        if base in vocab:
                            valid_words.append(t)
                            found_variant = True
                            break  # stop checking further
            if not found_variant:
                not_words.append(t)
            else:
                total_chars_count_from_valid_words = total_chars_count_from_valid_words + len(t)
    valid_word_count = len(valid_words)
    if len(tokens) == 0:
        percentage = 0
    else:
        percentage = 100 * len(valid_words) / len(tokens)
    return {"valid_words": valid_words, "not_words": not_words, "word_count":word_count, "valid_word_count":valid_word_count, "percentage": percentage, "total_chars_count_from_valid_words": total_chars_count_from_valid_words}


In [4]:
# Title 
df['% alphanum_standard title'] = df[title].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alpha_only title'] = df[title].apply(lambda x: re_pattern(x, alpha_only))
df['% en title'] = df[title].apply(percent_real_english_nltk)

# Description
df['% alphanum_standard descrip'] = df[descrip].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alpha_only descrip'] = df[descrip].apply(lambda x: re_pattern(x, alpha_only))
df['% en descrip'] = df[descrip].apply(percent_real_english_nltk)

In [ ]:
# Transcripts
# alphanumeric + standard symbols
# define what is alphanumeric / standard symbols (essentially opposite of special pattern from ocr_cleaning)
df['% alphanum_standard D_mcc_raw tr'] = df[og_tr].apply(lambda x: re_pattern(x, alphanum_standard))
df['% alphanum_standard D_mcc_cleaned tr'] = df[sc_tr].apply(lambda x: re_pattern(x, alphanum_standard))

# alphabet a-Z
df['% alpha_only D_mcc_raw tr'] = df[og_tr].apply(lambda x: re_pattern(x, alpha_only))
df['% alpha_only D_mcc_cleaned tr'] = df[sc_tr].apply(lambda x: re_pattern(x, alpha_only))

# # english words
df['% en D_mcc_raw tr'] = df[og_tr].apply(percent_real_english_nltk)
df['% en D_mcc_cleaned tr'] = df[sc_tr].apply(percent_real_english_nltk)


df[['og_valid_words', 'og_not_words', 'og_word_count', 'og_valid_word_count', 'og_percentage', 'og_total_chars_count_from_valid_words']] = df[og_tr].apply(
    lambda text: pd.Series(english_nltk_metrics(text, english_vocab))
)

df[['sc_v1_valid_words', 'sc_v1_not_words', 'sc_v1_word_count', 'sc_v1_valid_word_count', 'sc_v1_percentage', 'sc_v1_total_chars_count_from_valid_words']] = df[og_tr].apply(
    lambda text: pd.Series(english_nltk_metrics(text, english_vocab))
)

df[['sc_v2_valid_words', 'sc_not_words', 'sc_word_count', 'sc_valid_word_count', 'sc_percentage', 'sc_total_chars_count_from_valid_words']] = df[og_tr].apply(
    lambda text: pd.Series(english_nltk_metrics(text, english_vocab))
)


In [6]:
df[0:10]

,Title,Creator,Contributors,Date,Approximate Date,Source,Subject,Local Subject,S.C. County,Description,...,og_word_count,og_valid_word_count,og_percentage,og_total_chars_count_from_valid_words,sc_valid_words,sc_not_words,sc_word_count,sc_valid_word_count,sc_percentage,sc_total_chars_count_from_valid_words
0,Afro-American Newsboy Application signed by Mr...,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,An application to be a Newsboy for the Afro-Am...,...,35,35,100.000000,199,"[afro, american, newsboy, application, hereby,...",[],35,35,100.000000,199
1,Lighthouse Informer receipt,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A blank Lighthouse and Informer receipt.,...,18,18,100.000000,99,"[received, from, received, of, shedding, light...",[],18,18,100.000000,99
2,"The Lighthouse Solicitor's Record, Home Office...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,"A solicitor's record, home office order, and s...",...,68,68,100.000000,356,"[solicitor, record, name, address, city, state...",[],68,68,100.000000,356
3,The Lighthouse Remittance Envelope(Front),NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,The front of a Lighthouse remittance envelope.,...,9,9,100.000000,51,"[sender, address, shedding, light, for, growin...",[],9,9,100.000000,51
4,"Advertisment, Chicken Special at the Pig Trail...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,An ad about the Chicken Special for the Pig Tr...,...,33,32,96.969697,137,"[just, telephone, us, place, your, order, it, ...",[ll],33,32,96.969697,137
5,"The Lighthouse Solicitor's Record, Home Office...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,"A solicitor's record, home office order, and s...",...,68,68,100.000000,356,"[solicitor, record, name, address, city, state...",[],68,68,100.000000,356
6,The Lighthouse Prospectus,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A prospectus of the Lighthouse with permanent ...,...,96,96,100.000000,579,"[prospectus, permanent, publication, schedule,...",[],96,96,100.000000,579
7,The Lighthouse Temporary and Permanent Operati...,NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,Temporary and Permanent Operating Costs includ...,...,28,24,85.714286,168,"[temporary, operating, costs, office, printing...","[circ, perhanent, bookeeper, mgr]",28,24,85.714286,168
8,"Letter, Lighthouse Newspaper Sign-up Sheet sen...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A sign-up sheet for the Lighthouse Newspaper s...,...,37,37,100.000000,188,"[if, you, have, boy, interested, in, selling, ...",[],37,37,100.000000,188
9,"Letter, Lighthouse Newspaper Sign-up Sheet sen...",NaN,"McCray, John Henry, 1910-1987",NaN,NaN,Manuscripts; Accession 11294.,Lighthouse and Informer,NaN,NaN,A sign-up sheet for the Lighthouse Newspaper s...,...,37,37,100.000000,194,"[mr, kennedy, mrs, coleman, if, you, have, boy...",[],37,37,100.000000,194


In [7]:
# Save
save_csv(df, "new_metrics main -- all.csv")

In [8]:
# Averages
def column_stats(df, colname):
    """
    Returns basic statistics (min, mean, median, max, percentiles) for a numeric column.
    """
    if colname not in df.columns:
        raise ValueError(f"Column '{colname}' not found in DataFrame.")
    
    # Drop NaN values to avoid errors
    series = pd.to_numeric(df[colname], errors='coerce').dropna()

    if series.empty:
        return {"min": None, "p10": None, "p25": None, "mean": None, "median": None, "max": None}
    
    stats = {
        "min": series.min(),
        "p10": series.quantile(0.10),
        "p25": series.quantile(0.25),
        "mean": series.mean(),
        "median": series.median(),
        "max": series.max()
    }
    
    print(f"\nStats for column: **{colname}**")
    print("-" * 40)
    print(f"Minimum:        {stats['min']:.2f}")
    print(f"10th percentile: {stats['p10']:.2f}")
    print(f"25th percentile: {stats['p25']:.2f}")
    print(f"Mean:           {stats['mean']:.2f}")
    print(f"Median:         {stats['median']:.2f}")
    print(f"Maximum:        {stats['max']:.2f}")
    print("-" * 40)

    return stats

check_cols = [
    "% alphanum_standard title",
    "% alpha_only title",
    "% en title",
    "% alphanum_standard descrip",
    "% alpha_only descrip",
    "% en descrip",
    "% alphanum_standard D_mcc_raw tr",
    "% alphanum_standard D_mcc_cleaned tr",
    "% alpha_only D_mcc_raw tr",
    "% alpha_only D_mcc_cleaned tr",
    "% en D_mcc_raw tr",
    "% en D_mcc_cleaned tr"
]

for col in check_cols:
    column_stats(df, col)

print()


Stats for column: **% alphanum_standard title**
----------------------------------------
Minimum:        78.57
10th percentile: 100.00
25th percentile: 100.00
Mean:           99.98
Median:         100.00
Maximum:        100.00
----------------------------------------

Stats for column: **% alpha_only title**
----------------------------------------
Minimum:        0.00
10th percentile: 61.90
25th percentile: 66.33
Mean:           72.34
Median:         72.58
Maximum:        100.00
----------------------------------------

Stats for column: **% en title**
----------------------------------------
Minimum:        0.00
10th percentile: 100.00
25th percentile: 100.00
Mean:           99.73
Median:         100.00
Maximum:        100.00
----------------------------------------

Stats for column: **% alphanum_standard descrip**
----------------------------------------
Minimum:        0.00
10th percentile: 100.00
25th percentile: 100.00
Mean:           98.65
Median:         100.00
Maximum:      